# Ch.13 — Dimensionality Reduction

> **Running theme:** The real estate platform's 8-feature housing dataset lives in a space we cannot visualise. PCA compresses it by keeping maximum variance. t-SNE reveals cluster topology. UMAP balances speed, global structure, and the ability to transform new data. Each tool makes a different promise — know which promise was broken before trusting the picture.

## Core Idea

**PCA:** linear projection along eigenvectors of the covariance matrix, ordered by eigenvalue. Maximises retained variance. Invertible.

**t-SNE:** minimises KL divergence between high-d Gaussian and low-d Student-t similarity distributions. Preserves local neighbourhood topology. Distances between clusters are NOT meaningful.

**UMAP:** topological graph-matching. Faster than t-SNE, better global structure, has `transform()` for new points. The only one of the three suitable as a downstream preprocessing step.

## Running Example

Dataset: **California Housing** (sklearn) — 20,640 census districts, 8 features.  
All three methods project the 8-feature space down to 2D.  
Colour: `MedHouseVal` (continuous target, **not used for fitting** — purely for visual validation).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── Load and scale ─────────────────────────────────────────────────────────────
housing       = fetch_california_housing()
X             = housing.data          # (20640, 8)
y             = housing.target        # MedHouseVal — used only for colouring
feature_names = list(housing.feature_names)

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

print(f"Dataset: {X.shape[0]:,} samples × {X.shape[1]} features")
print("Features:", feature_names)

## PCA: Scree Plot and Explained Variance

In [ ]:
# ── Full PCA to inspect explained variance ─────────────────────────────────────
pca_full = PCA(n_components=X_sc.shape[1])
pca_full.fit(X_sc)

evr    = pca_full.explained_variance_ratio_
cumevr = evr.cumsum()
k95    = int((cumevr < 0.95).sum()) + 1   # fewest components for ≥95% variance

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

comp_idx = np.arange(1, len(evr) + 1)
ax1.bar(comp_idx, evr * 100, color='steelblue', alpha=0.8)
ax1.set_xlabel('Component'); ax1.set_ylabel('Explained variance (%)')
ax1.set_title('Scree plot — individual component variance')
ax1.set_xticks(comp_idx); ax1.grid(True, alpha=0.3)

ax2.plot(comp_idx, cumevr * 100, 'ko-', markersize=6)
ax2.axhline(y=95, color='r', linestyle='--', label='95% threshold')
ax2.axvline(x=k95, color='g', linestyle=':', label=f'k={k95} components')
ax2.set_xlabel('Number of components'); ax2.set_ylabel('Cumulative variance (%)')
ax2.set_title('Cumulative explained variance')
ax2.set_xticks(comp_idx); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\nComponents for ≥95% variance: {k95}")
for i, (ev, ce) in enumerate(zip(evr, cumevr), start=1):
    print(f"  PC{i}: {ev*100:5.1f}%  cumulative {ce*100:5.1f}%")

## PCA: Component Loadings (What Does Each PC Mean?)

In [ ]:
# ── Loadings heatmap ──────────────────────────────────────────────────────────
loadings = pca_full.components_[:4]   # first 4 PCs

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(loadings, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(feature_names))); ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.set_yticks(range(4)); ax.set_yticklabels([f'PC{i+1}' for i in range(4)])
plt.colorbar(im, ax=ax, label='Loading')
ax.set_title('PCA component loadings (first 4 PCs)')
plt.tight_layout(); plt.show()

print("PC1 dominated by:", feature_names[np.argmax(np.abs(loadings[0]))])
print("PC2 dominated by:", feature_names[np.argmax(np.abs(loadings[1]))])

## PCA: 2D Projection and Scatter Plot

In [ ]:
pca2  = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X_sc)
pct   = pca2.explained_variance_ratio_.sum() * 100

plt.figure(figsize=(8, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=2, alpha=0.4)
plt.colorbar(sc, label='MedHouseVal ($100k)')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.title(f'PCA 2D projection (retains {pct:.1f}% of variance)')
plt.tight_layout(); plt.show()

## PCA: Reconstruction Quality

PCA is invertible — the only method here that can reconstruct an approximation of the original data. Reconstruction error = variance of the dropped components.

In [ ]:
from sklearn.metrics import mean_squared_error

print("Reconstruction MSE vs number of components:")
for k in [1, 2, 3, 4, 6, 8]:
    pca_k   = PCA(n_components=k, random_state=42)
    X_proj  = pca_k.fit_transform(X_sc)
    X_recon = pca_k.inverse_transform(X_proj)   # back to d=8
    mse     = mean_squared_error(X_sc, X_recon)
    var_ret = pca_k.explained_variance_ratio_.sum() * 100
    print(f"  k={k}: MSE={mse:.4f}   variance retained={var_ret:.1f}%")

## t-SNE: 2D Projection

t-SNE is $O(n^2)$ naive; we first reduce to 30 PCA dims to remove noise and speed things up.  
**Warning:** distances between clusters in the t-SNE plot are NOT meaningful.

In [ ]:
# ── PCA pre-reduction ─────────────────────────────────────────────────────────
n_pca_pre = min(30, X_sc.shape[1])   # 8 features → use all 8
X_pca_pre = PCA(n_components=n_pca_pre, random_state=42).fit_transform(X_sc)

# ── t-SNE ─────────────────────────────────────────────────────────────────────
# On 20k rows this takes ~2–5 minutes; subsample if too slow
SAMPLE = 5000   # reduce further if needed
rng    = np.random.default_rng(42)
idx    = rng.choice(len(X_pca_pre), SAMPLE, replace=False)
X_sub  = X_pca_pre[idx]
y_sub  = y[idx]

tsne   = TSNE(n_components=2, perplexity=30, learning_rate='auto',
              init='pca', random_state=42, n_jobs=-1)
X_tsne = tsne.fit_transform(X_sub)
print(f"t-SNE embedding shape: {X_tsne.shape}  (sample of {SAMPLE:,} used)")

plt.figure(figsize=(8, 6))
sc = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sub, cmap='viridis', s=4, alpha=0.5)
plt.colorbar(sc, label='MedHouseVal ($100k)')
plt.xlabel('t-SNE dim 1'); plt.ylabel('t-SNE dim 2')
plt.title('t-SNE 2D projection (perplexity=30) — colour=MedHouseVal')
plt.tight_layout(); plt.show()

## t-SNE: Perplexity Sweep

Perplexity controls the effective neighbourhood size. Try multiple values and look for **consistent** cluster patterns across runs.

In [ ]:
perplexities = [5, 30, 80]
fig, axes    = plt.subplots(1, len(perplexities), figsize=(18, 5))

for ax, perp in zip(axes, perplexities):
    tsne_p = TSNE(n_components=2, perplexity=perp, learning_rate='auto',
                  init='pca', random_state=42, n_jobs=-1)
    Z = tsne_p.fit_transform(X_sub)
    ax.scatter(Z[:, 0], Z[:, 1], c=y_sub, cmap='viridis', s=3, alpha=0.5)
    ax.set_title(f'perplexity = {perp}')
    ax.tick_params(labelleft=False, labelbottom=False)

plt.suptitle('t-SNE: perplexity sweep — small=fragmented, large=diffuse', y=1.02)
plt.tight_layout(); plt.show()
print("Note: the DISTANCE between clusters is NOT meaningful in any of these plots.")

## UMAP: 2D Projection

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        random_state=42, n_jobs=-1)
    X_umap  = reducer.fit_transform(X_sc)   # use full dataset (UMAP is fast)

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap='viridis', s=2, alpha=0.4)
    plt.colorbar(sc, label='MedHouseVal ($100k)')
    plt.xlabel('UMAP dim 1'); plt.ylabel('UMAP dim 2')
    plt.title('UMAP 2D projection — colour=MedHouseVal')
    plt.tight_layout(); plt.show()

except ImportError:
    print("umap-learn not installed — run: pip install umap-learn")
    X_umap = None

## UMAP: n_neighbors Sweep

In [ ]:
try:
    import umap
    neighbors_vals = [5, 15, 50]
    fig, axes      = plt.subplots(1, len(neighbors_vals), figsize=(18, 5))

    for ax, nn in zip(axes, neighbors_vals):
        r = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1, random_state=42, n_jobs=-1)
        Z = r.fit_transform(X_sc)
        ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap='viridis', s=1, alpha=0.3)
        ax.set_title(f'n_neighbors = {nn}')
        ax.tick_params(labelleft=False, labelbottom=False)

    plt.suptitle('UMAP: n_neighbors sweep — small=local, large=global', y=1.02)
    plt.tight_layout(); plt.show()

except ImportError:
    print("pip install umap-learn")

## Side-by-Side Comparison: PCA vs t-SNE vs UMAP

In [ ]:
# Align on the same sample used for t-SNE (5k rows)
X_pca_sub = X_pca[idx]  # 2D PCA embedding of same 5k sample

have_umap = X_umap is not None
n_cols    = 3 if have_umap else 2
fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5))

methods = [('PCA', X_pca_sub), ('t-SNE', X_tsne)]
if have_umap:
    methods.append(('UMAP', X_umap[idx]))

for ax, (name, Z) in zip(axes, methods):
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=y_sub, cmap='viridis', s=4, alpha=0.5)
    plt.colorbar(sc, ax=ax, label='MedHouseVal')
    ax.set_title(name)
    ax.tick_params(labelleft=False, labelbottom=False)

plt.suptitle('PCA vs t-SNE vs UMAP — same 5k sample, coloured by MedHouseVal', y=1.02)
plt.tight_layout(); plt.show()

## What Can Go Wrong: t-SNE Distances Are Not Meaningful

Take two well-separated clusters in the t-SNE plot and check that their **true** mean feature vectors are similar.

In [ ]:
# Find two clusters by splitting t-SNE x-axis at the median
median_x = np.median(X_tsne[:, 0])
left_mask  = X_tsne[:, 0] < median_x
right_mask = ~left_mask

X_left  = X_sub[left_mask]
X_right = X_sub[right_mask]

# In ORIGINAL (scaled) feature space
left_mean  = X_left.mean(axis=0)
right_mean = X_right.mean(axis=0)
eucl_dist  = np.linalg.norm(left_mean - right_mean)

# In t-SNE space
tsne_left_center  = X_tsne[left_mask].mean(axis=0)
tsne_right_center = X_tsne[right_mask].mean(axis=0)
tsne_dist         = np.linalg.norm(tsne_left_center - tsne_right_center)

print("Left cluster centroid (scaled features):", np.round(left_mean[:4], 3))
print("Right cluster centroid (scaled features):", np.round(right_mean[:4], 3))
print(f"\nEuclidean distance in ORIGINAL 8-d space: {eucl_dist:.4f}")
print(f"Euclidean distance in t-SNE 2-d space:    {tsne_dist:.4f}")
print("\n→ t-SNE distances do NOT reflect original feature distances")
print("  Only the presence of clusters (topology) is meaningful.")

## What Can Go Wrong: Scaling Matters for PCA

Without standardisation, PCA is dominated by features with large raw variance — here `AveRooms` which has much wider range than `MedInc`.

In [ ]:
pca_raw    = PCA(n_components=2).fit(X)     # RAW (unscaled)
pca_scaled = PCA(n_components=2).fit(X_sc)  # STANDARDISED

print("PC1 loadings (raw vs standardised):")
header = "  ".join(f"{n:>12}" for n in feature_names)
print(f"  {'Feature':<12}  {header}")
for label, pca_obj in [('Raw      ', pca_raw), ('Scaled   ', pca_scaled)]:
    vals = "  ".join(f"{v:>12.4f}" for v in pca_obj.components_[0])
    print(f"  {label} {vals}")

print("\nRaw PCA: PC1 captures mostly high-variance features (AveRooms, AveBedrms, Population)")
print("Scaled PCA: PC1 is a more balanced combination of all features")

## Exercises

1. **UMAP for preprocessing.** Fit `UMAP(n_components=4)` on the training set, then use `reducer.transform(X_test)` to project the test set. Train a Ridge regression on the 4-UMAP features and compare RMSE to PCA with `n_components=4`. Which preprocessing loses less information?

2. **Reproducing t-SNE plots.** Run t-SNE three times with different `random_state` values (but the same `perplexity=30`). Do the same clusters appear each time? Are their relative positions preserved? What changes and what stays the same?

3. **PCA for compression.** Fit PCA on the training set and compute the reconstruction MSE for `k = 1, 2, 3, 4, 6, 8`. Plot MSE vs k. At what k does adding another component reduce MSE by less than 5%? Is that the same k as the 95% cumulative EVR heuristic?

In [ ]:
# Exercise 1 — UMAP for preprocessing
# TODO: your solution here

In [ ]:
# Exercise 2 — Reproducing t-SNE plots
# TODO: your solution here

In [ ]:
# Exercise 3 — PCA compression MSE vs k
# TODO: your solution here